# Prototype Development Report: BVG Construction Plans
### Title and Date Extraction
**BIP Project with BVG, HTW Berlin | Group 1 (Sicarios)**  

## 1. Introduction
This report describes the prototype built to extract the title and date from the title block (Schriftfeld) of scanned BVG construction plans. Within the team, I was responsible for the LLM part of the project: researching the zero-shot approach, connecting to the HTW inference API, designing the prompt, building the evaluation and the Streamlit demo, and preparing the fine-tuning pipeline.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

## 2. Approach
The prototype extracts the title and date by prompting a multimodal large language model accessed through the HTW Berlin inference API (Ollama server, OpenAI-compatible interface). The default model configured in the evaluation script is `qwen3.8-27b`. No model weights are trained or modified: the model is only prompted (zero-shot), not fine-tuned.

The prompt is domain-specific. It tells the model that the input is a German-language BVG construction plan, requires the title and date to be transcribed exactly as written (without translation), and explicitly instructs the model to return an empty date field rather than guess one when no date is present on the sheet.

## 3. Implementation
The following components were built:
* **`evaluate.py`**: Runs the prompt over the held-out test set and scores each prediction against manually verified ground truth.
* **`streamlit_app.py`**: A demo interface with single-image upload and gallery views.
* **`quick_test.py`**: A minimal script verifying API connectivity for a single image.
* **`analyze_failures.py`**: Cross-references failed predictions with ground-truth confidence notes.

## 4. Evaluation Setup & Results

In [2]:
import pandas as pd
from pathlib import Path

# Load evaluation results generated locally by evaluate.py
eval_csv_path = Path("eval_zero_shot.csv")

if eval_csv_path.exists():
    df = pd.read_csv(eval_csv_path)
    
    # Calculate confusion matrix components
    tp = (df["date_presence_class"] == "TP").sum()
    fn = (df["date_presence_class"] == "FN").sum()
    fp = (df["date_presence_class"] == "FP").sum()
    tn = (df["date_presence_class"] == "TN").sum()
    
    # Calculate performance metrics
    total = len(df)
    accuracy = (tp + tn) / total if total > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    mean_sim = df["title_similarity"].mean()

    # Build summary dataframe
    eval_summary = {
        "Metric": [
            "Confusion Matrix (TP / FN / FP / TN)", 
            "Precision (Date Detection)", 
            "Recall (Date Detection)", 
            "F1 Score (Date Detection)", 
            "Accuracy (Date Presence)", 
            "Mean Title Similarity"
        ],
        "Zero-Shot Run": [
            f"{tp} / {fn} / {fp} / {tn}", 
            round(precision, 2), 
            round(recall, 2), 
            round(f1, 2), 
            round(accuracy, 2), 
            round(mean_sim, 2)
        ]
    }
    
    df_eval = pd.DataFrame(eval_summary)
    
    # Output as a clean Markdown table
    print("### Evaluation Results (Dynamic)")
    display(df_eval)
else:
    print("Local 'eval_zero_shot.csv' not found. Please run evaluate.py first.")

Local 'eval_zero_shot.csv' not found. Please run evaluate.py first.


## 5. Transfer Learning (Planned, Not Executed)
As a next stage beyond zero-shot prompting, we would like to apply transfer learning: taking a pre-trained vision-language model that already reads documents well and adapting it to BVG plans with our own labeled data. The model would keep its general knowledge and learn the specific layout of the BVG title blocks, the German terminology used on the sheets, and the date formats that appear on them. This is expected to reduce the errors seen in Section 6, especially wrong titles and hallucinated dates.
A pipeline for this was already implemented using LoRA (Low-Rank Adaptation), a parameter-efficient transfer learning method that trains only a small set of additional weights instead of the whole model. It includes dataset preparation with cropping and augmentation, and a training configuration for Qwen2-VL-7B-Instruct using LLaMA-Factory. The code is included in the project repository.
However, transfer learning was not carried out because it is expensive:
- Hardware cost: training a 7-billion-parameter vision-language model requires a high-end NVIDIA GPU with CUDA support. This was not available on the project laptops, and the HTW inference API is inference-only and cannot run training jobs. Renting cloud GPUs or reserving time on a GPU cluster adds cost.
- Data cost: reliable results need more labeled plans than the current dataset. Every additional plan must be labeled by hand (title, date and title-block position), which takes considerable time.
- Time cost: each training run, followed by evaluation and tuning, takes hours of GPU time and has to be repeated several times.
With the small dataset currently available, the gain over zero-shot prompting is also uncertain. For this reason, the zero-shot prototype was prioritised, and transfer learning remains the planned next step if GPU resources and budget become available.


## 6. Limitations

- The test set is small (9 plans), so all figures have high uncertainty and should be read as indicative, not final.
- Results vary between runs of the same prompt and model (see Section 5.3).
- Two of the 18 predictions in Run 1 are missing because of API timeouts.
- Crop quality depends on the manually labeled bounding boxes, which may not always match the title block exactly.


## 7. Conclusion and Next Steps
The zero-shot prototype works end to end: it extracts titles and dates from real BVG plans, can be evaluated against ground truth, and can be demonstrated through the Streamlit interface. In the latest run it correctly decided whether a date was present in 13 of 18 images (accuracy 0.72), with precision 0.60, recall 0.50 and F1 0.55 for date detection, and a mean title similarity of 0.65 in the first run. This is a usable baseline, but not yet reliable enough for production use. The main weaknesses are missed dates, occasional hallucinated dates, and errors on title-block crops rather than on full pages.
Recommended next steps:
- Enlarge the test set and repeat each evaluation several times, reporting the average, to obtain stable figures.
- Compare full-page and crop results separately, and review the crop alignment of the failing plans. If crops remain worse, use full pages only.
- Refine the prompt based on the failure cases.
